<a href="https://colab.research.google.com/github/fdac25/MP2/blob/main/Vis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MiniProject 2 — Part 3: Inactivity Gap Investigation

This notebook investigates the longest inactivity gap in each assigned software project. It examines commit messages before and after each gap, identifies the main development themes, and uses the available evidence to explore possible explanations for inactivity and subsequent activity.

The analysis uses the commit data collected in Part 1 and the inactivity-gap statistics calculated in Part 2.

## 1. Load Commit Data and Project Statistics

This section loads the commit-level dataset collected in Part 1 and the project statistics generated in Part 2.

The commit dataset contains project identifiers, commit hashes, authors, timestamps, and commit messages. The project statistics provide the start, end, and duration of each project's longest inactivity gap.

These datasets form the basis for investigating development activity before and after each gap.

In [1]:
import pandas as pd

commits = pd.read_csv(
    "aalkafou_project_summary.csv",
    sep=";"
)

stats = pd.read_csv(
    "aalkafou_project_stats.csv",
    sep=";"
)

print("Total commits:", len(commits))
print("Projects:", commits["project_wocid"].nunique())
print("Project statistics:", len(stats))

display(commits.head())
display(stats.head())

Total commits: 42980
Projects: 10
Project statistics: 10


,project_wocid,commit_sha1,author,time,commit message
0,knygren_glmbayes,00072e026eadd6ffc3482d026f501f92577bbcd1,Nygren <Kjell.Nygren@sanofi.com>,1596427257,Finished most cleanup for the new rIndependent...
1,knygren_glmbayes,00489b17e85cab12b6a3624ed9ac3df8c202e1e1,knygren <kjell.a.nygren@gmail.com>,1752529616,Add Initial News.MD for version tracking\n
2,knygren_glmbayes,0096d32f114cc9233dd817f1df47343073340294,knygren <kjell.a.nygren@gmail.com>,1757498572,Adding Rdpack capabilities to ensure enabled b...
3,knygren_glmbayes,0102d1080849964f0ee67b9815bffbef1ad34b02,knygren <kjell.a.nygren@gmail.com>,1753958575,Adds PKG_CPP_FLAGS to configure script output.\n
4,knygren_glmbayes,0269eb3d3172fe0038f54f249148737b2c75ecb2,knygren <kjell.a.nygren@gmail.com>,1753981278,Additional edits to configure script\n


,Project,ncommits,nauthors,from,to,nstars,nforks,lastGHCommitDate,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,ActivityPattern,NumberOfGapsInTimeline
0,ajdawson_windspharm,366,23,2012-06-16,2025-10-01,100,39,2026-08-25,2018-09,2021-01,29,45,declining,13
1,cselab_aphros,10702,8,2017-11-28,2025-07-20,460,52,2025-07-20,2024-08,2025-03,8,2,declining,3
2,epics-base_pva2pva,660,19,2015-09-04,2025-07-28,5,15,2026-08-02,2023-04,2023-11,8,28,declining,8
3,geos-esm_mapl,14975,99,2019-06-07,2026-04-16,45,25,2026-09-23,NaN,NaN,0,0,rising,0
4,icbi-lab_luca,578,8,2020-10-29,2025-01-19,94,30,2025-01-19,2023-12,2024-11,12,5,declining,2


## 2. Extract Commits Before and After Inactivity Gaps

For each project with an identified inactivity gap, the last 10 commits before the gap and the first 10 commits after the gap are extracted in chronological order.

If fewer than 10 commits are available on either side, all available commits are included. Projects without an inactivity gap are excluded from this comparison.

The extracted records are labeled `Before` or `After` and saved to `aalkafou_project_gap_commits.csv` for subsequent thematic analysis.

In [2]:
# Convert Unix timestamps to dates
commits["date"] = pd.to_datetime(
    commits["time"], unit="s", utc=True
)

gap_commits = []

for _, row in stats.iterrows():
    project = row["Project"]

    # Projects without inactivity gaps cannot have
    # a before/after gap comparison.
    if pd.isna(row["LongestGapStart"]) or row["LongestGapLength"] == 0:
        print(f"{project}: No inactivity gap — skipped")
        continue

    gap_start = pd.to_datetime(row["LongestGapStart"], utc=True)
    gap_end = pd.to_datetime(row["LongestGapEnd"], utc=True)
    gap_end_exclusive = gap_end + pd.DateOffset(months=1)

    project_commits = (
        commits[commits["project_wocid"] == project]
        .sort_values("date")
    )

    # Last 10 commits before the gap
    before = project_commits[
        project_commits["date"] < gap_start
    ].tail(10).copy()

    # First 10 commits after the gap
    after = project_commits[
        project_commits["date"] >= gap_end_exclusive
    ].head(10).copy()

    before["Period"] = "Before"
    after["Period"] = "After"

    gap_commits.extend([before, after])

    print(
        f"{project}: "
        f"{len(before)} before, "
        f"{len(after)} after"
    )

gap_commits_df = pd.concat(gap_commits, ignore_index=True)

gap_commits_df = gap_commits_df[
    [
        "Period",
        "project_wocid",
        "commit_sha1",
        "author",
        "time",
        "commit message"
    ]
]

gap_commits_df.to_csv(
    "aalkafou_project_gap_commits.csv",
    sep=";",
    index=False
)

print("\nTotal extracted commits:", len(gap_commits_df))
display(gap_commits_df.head())

ajdawson_windspharm: 10 before, 10 after
cselab_aphros: 10 before, 2 after
epics-base_pva2pva: 10 before, 10 after
geos-esm_mapl: No inactivity gap — skipped
icbi-lab_luca: 10 before, 5 after
knygren_glmbayes: 10 before, 10 after
learning-at-home_hivemind: 10 before, 10 after
nfft_nfft: 10 before, 10 after
rapidsai_cuxfilter: 10 before, 10 after
sonvt1710_deepface: 10 before, 10 after

Total extracted commits: 167


,Period,project_wocid,commit_sha1,author,time,commit message
0,Before,ajdawson_windspharm,ba30655fa521955075feb31b9ff76e924ebc3f03,Andrew Dawson <ajdawson@acm.org>,1534847247,Use tuple indexing for numpy arrays\n
1,Before,ajdawson_windspharm,940af958cfab4d5ebbae8e82a28c680d50c016dd,Andrew Dawson <ajdawson@acm.org>,1534847269,Update the changelog for v1.7.0\n
2,Before,ajdawson_windspharm,2e3a387dc5ebbc2bdb7fbac5f97ca270fdc99577,Andrew Dawson <ajdawson@acm.org>,1534847877,List version numbers with most recent first\n
3,Before,ajdawson_windspharm,fac40d7fad37f49c727472efb40a9ee548b865f8,Travis CI User <travis@example.org>,1534849039,Update docs after building Travis build 344 of...
4,Before,ajdawson_windspharm,aa6243c717ed3941b6bd4206cf04aaf466b8f842,Andrew Dawson <ajdawson@acm.org>,1534852776,Remove pip install instructions\n\npyspharm is...


## 3. Commit Message Analysis

This section examines the commit messages immediately before and after each project's longest inactivity gap.

Commit messages provide evidence of development activities, including feature development, bug fixes, documentation updates, dependency updates, releases, and automated contributions.

The messages are grouped by project and period to identify the two dominant development themes before and after inactivity.

In [3]:
for project in stats["Project"]:
    project_data = gap_commits_df[
        gap_commits_df["project_wocid"] == project
    ]

    if project_data.empty:
        continue

    print("\n" + "=" * 75)
    print("PROJECT:", project)
    print("=" * 75)

    for period in ["Before", "After"]:
        print(f"\n--- {period.upper()} GAP ---")

        period_data = project_data[
            project_data["Period"] == period
        ]

        for _, commit in period_data.iterrows():
            print(
                f"{commit['commit_sha1'][:8]} | "
                f"{str(commit['commit message']).strip()[:250]}"
            )


PROJECT: ajdawson_windspharm

--- BEFORE GAP ---
ba30655f | Use tuple indexing for numpy arrays
940af958 | Update the changelog for v1.7.0
2e3a387d | List version numbers with most recent first
fac40d7f | Update docs after building Travis build 344 of
ajdawson/windspharm

The docs were built from the tag 'v1.7.0rc1' against the commit
940af958cfab4d5ebbae8e82a28c680d50c016dd.

The Travis build that generated this commit is at
https://travis-ci.org/ajd
aa6243c7 | Remove pip install instructions

pyspharm is no longer available to install via pip, so omit the
instructions.
d25a5348 | Use the README as the package description
8524f73d | Update docs after building Travis build 346 of
ajdawson/windspharm

The docs were built from the tag 'v1.7.0rc2' against the commit
d25a5348245c08cfa4be93a329cebf334b0a2ea3.

The Travis build that generated this commit is at
https://travis-ci.org/ajd
a4a29721 | Merge remote-tracking branch 'origin/v1.7.x'
35b17249 | Update docs after building Travis build

## 4. Development Theme Classification

The commit messages before and after each project's longest inactivity gap were manually reviewed to identify the two most prominent development themes.

Themes were selected from the categories Feature development, Bug fixes, Documentation updates, Dependency updates, Release, Security patches, Automated bot contributions, and Other.

The classifications are based on the available commit messages rather than assumed project activity. Merge commits were considered in context to avoid treating them as separate development themes.

In [4]:
# Two dominant development themes before and after each gap
# Based on manual review of the extracted commit messages

project_themes = {
    "ajdawson_windspharm": {
        "BeforeThemes": "Documentation updates:Release",
        "AfterThemes": "Bug fixes:Feature development"
    },
    "cselab_aphros": {
        "BeforeThemes": "Bug fixes:Feature development",
        "AfterThemes": "Dependency updates:Feature development"
    },
    "epics-base_pva2pva": {
        "BeforeThemes": "Feature development:Bug fixes",
        "AfterThemes": "Release:Bug fixes"
    },
    "icbi-lab_luca": {
        "BeforeThemes": "Feature development:Dependency updates",
        "AfterThemes": "Bug fixes:Documentation updates"
    },
    "knygren_glmbayes": {
        "BeforeThemes": "Feature development:Bug fixes",
        "AfterThemes": "Dependency updates:Documentation updates"
    },
    "learning-at-home_hivemind": {
        "BeforeThemes": "Bug fixes:Documentation updates",
        "AfterThemes": "Dependency updates:Bug fixes"
    },
    "nfft_nfft": {
        "BeforeThemes": "Release:Bug fixes",
        "AfterThemes": "Feature development:Bug fixes"
    },
    "rapidsai_cuxfilter": {
        "BeforeThemes": "Feature development:Bug fixes",
        "AfterThemes": "Dependency updates:Feature development"
    },
    "sonvt1710_deepface": {
        "BeforeThemes": "Feature development:Bug fixes",
        "AfterThemes": "Bug fixes:Dependency updates"
    }
}

# Add themes to the project statistics
stats["BeforeThemes"] = stats["Project"].map(
    lambda p: project_themes.get(p, {}).get("BeforeThemes", "N/A")
)

stats["AfterThemes"] = stats["Project"].map(
    lambda p: project_themes.get(p, {}).get("AfterThemes", "N/A")
)

display(stats[["Project", "BeforeThemes", "AfterThemes"]])

,Project,BeforeThemes,AfterThemes
0,ajdawson_windspharm,Documentation updates:Release,Bug fixes:Feature development
1,cselab_aphros,Bug fixes:Feature development,Dependency updates:Feature development
2,epics-base_pva2pva,Feature development:Bug fixes,Release:Bug fixes
3,geos-esm_mapl,N/A,N/A
4,icbi-lab_luca,Feature development:Dependency updates,Bug fixes:Documentation updates
5,knygren_glmbayes,Feature development:Bug fixes,Dependency updates:Documentation updates
6,learning-at-home_hivemind,Bug fixes:Documentation updates,Dependency updates:Bug fixes
7,nfft_nfft,Release:Bug fixes,Feature development:Bug fixes
8,rapidsai_cuxfilter,Feature development:Bug fixes,Dependency updates:Feature development
9,sonvt1710_deepface,Feature development:Bug fixes,Bug fixes:Dependency updates


## 5. Investigating Possible Reasons for Inactivity

This section investigates possible explanations for each project's longest inactivity gap.

The analysis uses commit messages as primary evidence, supplemented by GitHub repository information, issues, pull requests, and project documentation where available.

For each project, the investigation considers:

- The development activities immediately before the gap.
- The changes introduced when contributions resumed.
- Possible reasons for the interruption.
- Whether the available evidence supports a specific explanation.

Hypothesized reasons are distinguished from confirmed facts. A period without commits does not necessarily mean that development was abandoned.

### 5.1 ajdawson_windspharm

**Longest inactivity gap:** September 2018 – January 2021 (29 months).

**Commit evidence:** Before the gap, commits focused on the version 1.7.0 release, changelog updates, and documentation builds. After the gap, development resumed with new `getuv` methods, code cleanup, documentation improvements, and NumPy compatibility fixes.

**Release evidence:** Version 1.7.0 was published on August 21, 2018, immediately before the gap. Version 2.0.0 was subsequently published on November 20, 2024, confirming that the project continued to receive major updates.

**GitHub issue evidence:** Issue #109, opened in June 2019 during the gap, reported unexpected divergence calculations. This demonstrates that users continued reporting technical concerns during the period of commit inactivity.

**Hypothesized gap reason:** The gap may reflect reduced development activity following the 1.7.0 release. However, the examined issues, commits, and release history do not confirm the underlying cause. Maintainer availability or changing priorities remain possible but unverified explanations.

**Conclusion:** The project recovered from its longest inactivity gap. Subsequent development focused on new functionality, maintenance, and compatibility. The evidence supports a period of reduced recorded development rather than confirmed abandonment.

**Evidence sources:**
- [GitHub repository](https://github.com/ajdawson/windspharm)
- [GitHub issue #109](https://github.com/ajdawson/windspharm/issues/109)
- [PyPI release history](https://pypi.org/project/windspharm/)

### 5.2 cselab_aphros

**Longest inactivity gap:** August 2024 – March 2025 (8 months).

**Commit evidence:** Before the gap, development focused on fixing build and linker errors, improving CMake configurations, and adding simulation examples. After the gap, only two commits were recorded: an update to the minimum CMake version and a new wave simulation example.

**GitHub issue evidence:** Several issues were opened during the gap, including #20 (visualizing simulation outputs), #23 (boundary problems), #24 (geometry and convergence), and #25 (complex geometry modeling). This shows that users continued reporting problems and asking questions despite the absence of recorded commits.

**Pull request evidence:** The repository currently shows no open pull requests and seven closed pull requests. This does not establish whether any proposed changes were stalled during the gap.

**Project documentation:** The README describes Aphros as research software developed by researchers associated with ETH Zurich and Harvard University. Its documented focus is computational fluid dynamics and multiphase-flow simulation.

**Hypothesized gap reason:** The interruption may reflect reduced development effort or changing research priorities. However, neither the examined issues nor the project documentation confirms a specific cause. The absence of commits should not be interpreted as evidence that the software was abandoned.

**Conclusion:** Development resumed after the gap, but only two subsequent commits appear in the analyzed dataset. User interest continued during the interruption, while the available evidence is insufficient to determine why development slowed.

**Evidence sources:**
- [GitHub repository and README](https://github.com/cselab/aphros)
- [GitHub issues](https://github.com/cselab/aphros/issues)
- [Issue #25 — Modeling questions](https://github.com/cselab/aphros/issues/25)
- [GitHub pull requests](https://github.com/cselab/aphros/pulls)

### 5.3 epics-base_pva2pva

**Longest inactivity gap:** April 2023 – November 2023 (8 months).

**Commit evidence:** Before the gap, contributions focused on supporting metadata in arrays of structures, fixing `dbLoadGroup`, adding examples, and adjusting tests. After the gap, development resumed with release preparation, compiler-warning fixes, and CI dependency updates.

**Release evidence:** The official release notes confirm that version 1.4.1 was released in December 2023. It included fixes for `dbLoadGroup` and support for top-level `meta` members in arrays of structures. These changes correspond directly to commits recorded before the gap.

**Project documentation:** The repository contains two components: QSRV, a PVAccess server integrated into EPICS IOCs, and p2p, a gateway/proxy. The README states that the p2p gateway is deprecated in favor of `p4p.gw`. This indicates a change in the project's supported functionality, but it does not establish the cause of the 2023 inactivity gap.

**GitHub issues and pull requests:** The repository has issue and pull-request histories, but the available evidence does not establish a specific discussion explaining the interruption.

**Hypothesized gap reason:** The interruption may reflect reduced development activity between releases or maintenance scheduling within the larger EPICS ecosystem. The December 2023 release suggests that development resumed around a release cycle, but there is insufficient evidence to confirm why the gap occurred.

**Conclusion:** Development recovered after the gap, culminating in version 1.4.1. The available evidence supports a release-oriented resumption of work rather than confirmed project abandonment.

**Evidence sources:**
- https://github.com/epics-base/pva2pva
- https://github.com/epics-base/pva2pva/issues
- https://github.com/epics-base/pva2pva/pulls
- https://epics-base.github.io/pva2pva/release_notes.html
- https://github.com/epics-base/pva2pva/blob/master/README.md

### 5.4 icbi-lab_luca

**Longest inactivity gap:** December 2023 – November 2024 (12 months).

**Commit evidence:** Before the gap, development focused on integrating Harmony and MNN, modifying pipeline configurations, removing problematic dependencies, and updating documentation. After the gap, contributions focused on README updates and correcting EGFR mutation information in patient metadata and the exported atlas.

**Development transition:** The work shifted from implementing and configuring analysis functionality toward documentation and data-correction tasks. Only five commits were recorded after the gap in the analyzed dataset.

**GitHub issues and pull requests:** The extracted commit history references pull requests associated with Harmony integration, README updates, and EGFR mutation corrections. These demonstrate collaborative changes but do not establish why the project experienced a year without commits.

**Hypothesized gap reason:** The interruption may reflect a transition from active pipeline development to maintenance of research outputs, or reduced activity after an initial development phase. These are hypotheses rather than confirmed explanations.

**Conclusion:** Development resumed after the 12-month gap, but the subsequent activity was limited and focused primarily on documentation and correcting scientific metadata. The evidence is insufficient to establish the underlying cause of the interruption.

**Evidence sources:**
- https://github.com/icbi-lab/luca
- https://github.com/icbi-lab/luca/issues
- https://github.com/icbi-lab/luca/pulls

### 5.5 knygren_glmbayes

**Longest inactivity gap:** September 2020 – June 2025 (58 months).

**Commit evidence:** Before the gap, development focused on simplifying and refactoring simulation code, particularly the `rindependent_Normal_Gamma` functionality and Gamma sampling. After the gap, commits focused on compatibility with newer R versions, package documentation, RcppParallel support, and preparation for CRAN submission.

**Development transition:** The activity shifted from implementing and refining statistical functionality toward modernizing the package and preparing it for distribution.

**GitHub issues and pull requests:** The commit messages provide evidence of renewed package maintenance, but they do not establish whether a particular issue, pull request, or maintainer decision triggered the restart. The issue and pull-request histories require further verification.

**Hypothesized gap reason:** The extended inactivity may reflect reduced maintenance priority or a pause in active development. The renewed focus on R compatibility and CRAN preparation suggests an effort to modernize the package after the interruption. However, no confirmed explanation for the gap has been established.

**Conclusion:** Development resumed after nearly five years of recorded inactivity. The post-gap commits indicate a substantial renewed maintenance effort, but the available evidence does not explain why development originally stopped.

**Evidence sources:**
- https://github.com/knygren/glmbayes
- https://github.com/knygren/glmbayes/issues
- https://github.com/knygren/glmbayes/pulls

### 5.6 learning-at-home_hivemind

**Longest inactivity gap:** April 2024 (1 month).

**Commit evidence:** Before the gap, development included Windows compatibility improvements, dependency fixes, README corrections, and updates to machine-learning examples. After the gap, contributions addressed Pydantic v2 compatibility, PyTorch API changes, gradient-scaler imports, testing, and dependency maintenance.

**Development transition:** The commits before and after the gap both indicate ongoing software maintenance. After the gap, compatibility with evolving machine-learning dependencies became particularly prominent.

**GitHub issues and pull requests:** The extracted commits include merged contributions and dependency-related fixes. However, no specific issue or pull request has been verified as explaining the one-month interruption.

**Hypothesized gap reason:** The gap may represent normal variation in development activity rather than a significant interruption. There is insufficient evidence to attribute it to a technical problem, organizational change, or maintainer availability.

**Conclusion:** The project resumed contributions after a brief pause. The available evidence suggests continued maintenance rather than a meaningful period of abandonment.

**Evidence sources:**
- https://github.com/learning-at-home/hivemind
- https://github.com/learning-at-home/hivemind/issues
- https://github.com/learning-at-home/hivemind/pulls

### 5.7 nfft_nfft

**Longest inactivity gap:** March 2023 – March 2024 (13 months).

**Commit evidence:** Before the gap, development focused on preparing version 3.5.4, updating the changelog, maintaining GitHub workflows, improving documentation, and fixing numerical computation issues.

**After the gap:** Development resumed with a new radial basis function kernel, improvements to Julia and MATLAB interfaces, numerical fixes, and support for ARM-based Mac systems.

**Release evidence:** The pre-gap commits reference the preparation and release of version 3.5.4. This suggests that the interruption occurred after a release-related development period.

**Development transition:** The work shifted from release preparation and maintenance toward new numerical functionality, platform compatibility, and bug fixes.

**GitHub issues and pull requests:** The commit history does not identify a specific issue or pull request explaining the inactivity. The repository's issue and pull-request histories have not yet been independently verified for an explanation.

**Hypothesized gap reason:** The interruption may reflect a pause in development following version 3.5.4 or reduced development priority between releases. However, the available evidence does not confirm either explanation.

**Conclusion:** The project resumed development after a 13-month gap, introducing new functionality and compatibility improvements. The cause of the interruption remains unconfirmed.

**Evidence sources:**
- https://github.com/NFFT/nfft
- https://github.com/NFFT/nfft/issues
- https://github.com/NFFT/nfft/pulls
- https://github.com/NFFT/nfft/releases

### 5.8 rapidsai_cuxfilter

**Longest inactivity gap:** January 2019 (1 month).

**Commit evidence:** Before the gap, development focused on Docker configuration, slider interactions, server-session fixes, cuDF compatibility, and Jupyter notebook examples.

**After the gap:** Contributions resumed with updates for cuDF 0.5.0, nginx reverse-proxy configuration, server ports, and README documentation.

**Development transition:** Both periods focused on maintaining and improving the visualization application's infrastructure. The post-gap commits placed additional emphasis on compatibility with newer RAPIDS dependencies and deployment configuration.

**GitHub issues and pull requests:** The extracted commit messages do not identify a specific issue or pull request explaining the interruption. The issue and pull-request histories have not yet been independently verified for a cause.

**Hypothesized gap reason:** The one-month interruption may reflect normal variation in development activity or the timing of dependency updates. There is insufficient evidence to establish a specific cause.

**Conclusion:** Development continued after a brief period without recorded commits. The surrounding activity suggests ongoing maintenance and adaptation to the evolving RAPIDS ecosystem, rather than abandonment.

**Evidence sources:**
- https://github.com/rapidsai/cuxfilter
- https://github.com/rapidsai/cuxfilter/issues
- https://github.com/rapidsai/cuxfilter/pulls
- https://github.com/rapidsai/cuxfilter/releases

### 5.9 sonvt1710_deepface

**Longest inactivity gap:** October 2020 (1 month).

**Commit evidence:** Before the gap, development focused on face-detection functionality, MTCNN integration, TensorFlow/Keras compatibility, bug fixes, and release-related changes.

**After the gap:** Contributions resumed with improvements to face detectors, age-prediction preprocessing, documentation, LightGBM requirements, and faster MTCNN inference.

**Development transition:** Both periods focused on improving facial-analysis functionality and maintaining compatibility with machine-learning dependencies. The post-gap commits included additional performance and preprocessing improvements.

**GitHub issues and pull requests:** The extracted commits show ongoing technical improvements, but they do not identify a specific issue or pull request explaining the one-month interruption. No independent explanation has yet been verified from the issue or PR history.

**Hypothesized gap reason:** The short interruption may represent normal fluctuations in contribution frequency. There is insufficient evidence to associate it with a technical problem, organizational change, or reduced maintainer availability.

**Conclusion:** Development resumed after a brief gap, with continued improvements to detection, preprocessing, and dependency compatibility. The evidence does not support interpreting this period as project abandonment.

**Evidence sources:**
- https://github.com/sonvt1710/deepface
- https://github.com/sonvt1710/deepface/issues
- https://github.com/sonvt1710/deepface/pulls
- https://github.com/sonvt1710/deepface/releases

### 5.10 geos-esm_mapl

**Longest inactivity gap:** None detected (0 months).

**Commit evidence:** The monthly commit time series showed continuous activity, with at least one recorded commit in every month of the analyzed project history.

**Before and after comparison:** Not applicable. Since no zero-commit month was detected, there is no inactivity gap around which to extract before-and-after commits.

**GitHub issues and pull requests:** A gap-specific investigation is not applicable. Issues and pull requests may provide information about ongoing development, but they cannot explain an inactivity gap that was not observed.

**Hypothesized gap reason:** Not applicable.

**Conclusion:** This project serves as a useful comparison case because it maintained continuous monthly commit activity throughout the analyzed period. Its development themes before and after a gap are recorded as `N/A`.

**Evidence source:**
- https://github.com/GEOS-ESM/MAPL

In [8]:
# Save Part 3 theme classifications to project statistics
stats.to_csv("aalkafou_project_stats.csv", sep=";", index=False)

# Verify the saved file
saved_stats = pd.read_csv("aalkafou_project_stats.csv", sep=";")

print("Projects:", len(saved_stats))
print("BeforeThemes column:", "BeforeThemes" in saved_stats.columns)
print("AfterThemes column:", "AfterThemes" in saved_stats.columns)

display(saved_stats[["Project", "BeforeThemes", "AfterThemes"]])

Projects: 10
BeforeThemes column: True
AfterThemes column: True


,Project,BeforeThemes,AfterThemes
0,ajdawson_windspharm,Documentation updates:Release,Bug fixes:Feature development
1,cselab_aphros,Bug fixes:Feature development,Dependency updates:Feature development
2,epics-base_pva2pva,Feature development:Bug fixes,Release:Bug fixes
3,geos-esm_mapl,NaN,NaN
4,icbi-lab_luca,Feature development:Dependency updates,Bug fixes:Documentation updates
5,knygren_glmbayes,Feature development:Bug fixes,Dependency updates:Documentation updates
6,learning-at-home_hivemind,Bug fixes:Documentation updates,Dependency updates:Bug fixes
7,nfft_nfft,Release:Bug fixes,Feature development:Bug fixes
8,rapidsai_cuxfilter,Feature development:Bug fixes,Dependency updates:Feature development
9,sonvt1710_deepface,Feature development:Bug fixes,Bug fixes:Dependency updates


In [9]:
from pathlib import Path

required_files = [
    "aalkafou_project_gap_commits.csv",
    "aalkafou_project_stats.csv",
    "aalkafou_gap_analysis.ipynb",
]

for filename in required_files:
    path = Path(filename)
    print(f"{'OK' if path.exists() else 'MISSING'}: {filename}")

gap_data = pd.read_csv("aalkafou_project_gap_commits.csv", sep=";")

print("\nExtracted commits:", len(gap_data))
print("Projects with extracted commits:", gap_data["project_wocid"].nunique())
print("Projects in statistics:", len(saved_stats))

OK: aalkafou_project_gap_commits.csv
OK: aalkafou_project_stats.csv
OK: aalkafou_gap_analysis.ipynb

Extracted commits: 167
Projects with extracted commits: 9
Projects in statistics: 10
